# Notebook 03 — Whitespace Analysis

**Project:** APAC Territory Planning  
**Objective:** Identify unassigned and never-touched accounts, quantify the revenue upside, and cluster whitespace accounts into priority tiers using K-means.

**Business Question:** Where is the whitespace — and which accounts should we go after first?

In [1]:
import pandas as pd
import duckdb
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline
import numpy as np
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

DATA_DIR = "../data/raw"

In [2]:
accounts      = pd.read_csv(f"{DATA_DIR}/accounts.csv")
reps          = pd.read_csv(f"{DATA_DIR}/reps.csv")
assignments   = pd.read_csv(f"{DATA_DIR}/assignments.csv")
opportunities = pd.read_csv(f"{DATA_DIR}/opportunities.csv")
customers     = pd.read_csv(f"{DATA_DIR}/customers.csv")

con = duckdb.connect()
con.register("accounts",      accounts)
con.register("reps",          reps)
con.register("assignments",   assignments)
con.register("opportunities", opportunities)
con.register("customers",     customers)

print("Tables registered:")
print(con.execute("SHOW TABLES").df().to_string(index=False))

Tables registered:
         name
     accounts
  assignments
    customers
opportunities
         reps


## 1. Identify Whitespace Accounts

In [3]:
sql = """
SELECT
    a.account_id,
    a.company_name,
    a.country,
    a.subregion,
    a.industry,
    a.employee_band,
    a.estimated_arr,
    a.segment,
    a.account_status,
    asgn.coverage_status,
    asgn.engagement_status,
    COALESCE(asgn.last_activity_date, 'Never') AS last_activity_date
FROM accounts a
JOIN assignments asgn ON a.account_id = asgn.account_id
WHERE a.is_customer = 0
  AND (
      asgn.coverage_status = 'Unassigned'
      OR asgn.engagement_status IN ('Stale', 'No Coverage')
  )
"""

with open("../sql/whitespace_accounts.sql", "w") as f:
    f.write(sql)

whitespace = con.execute(sql).df()

print(f"Total whitespace accounts: {len(whitespace):,}")
print()
print("BY SUBREGION")
print(whitespace.groupby("subregion").size().sort_values(ascending=False).to_string())
print()
print("BY ENGAGEMENT STATUS")
print(whitespace.groupby("engagement_status").size().sort_values(ascending=False).to_string())

Total whitespace accounts: 3,462

BY SUBREGION
subregion
SEA              1016
Greater China     893
India             656
North Asia        618
ANZ               279

BY ENGAGEMENT STATUS
engagement_status
Stale          2826
No Coverage     636


## 2. TAM vs Actual ARR by Subregion

In [4]:
sql = """
WITH tam AS (
    SELECT
        subregion,
        SUM(estimated_arr)                          AS tam_arr,
        COUNT(account_id)                           AS total_accounts
    FROM accounts
    GROUP BY subregion
),
actual AS (
    SELECT
        a.subregion,
        SUM(c.arr)                                  AS actual_arr,
        COUNT(DISTINCT c.customer_id)               AS n_customers
    FROM accounts a
    JOIN customers c ON a.account_id = c.account_id
    GROUP BY a.subregion
),
ws AS (
    SELECT
        a.subregion,
        SUM(a.estimated_arr)                        AS whitespace_arr,
        COUNT(a.account_id)                         AS whitespace_accounts
    FROM accounts a
    JOIN assignments asgn ON a.account_id = asgn.account_id
    WHERE a.is_customer = 0
      AND (asgn.coverage_status = 'Unassigned'
           OR asgn.engagement_status IN ('Stale', 'No Coverage'))
    GROUP BY a.subregion
)
SELECT
    t.subregion,
    t.tam_arr,
    COALESCE(a.actual_arr, 0)                       AS actual_arr,
    COALESCE(a.actual_arr, 0) / t.tam_arr * 100     AS penetration_pct,
    COALESCE(w.whitespace_arr, 0)                   AS whitespace_arr,
    COALESCE(w.whitespace_accounts, 0)              AS whitespace_accounts
FROM tam t
LEFT JOIN actual a ON t.subregion = a.subregion
LEFT JOIN ws w ON t.subregion = w.subregion
ORDER BY penetration_pct DESC
"""

with open("../sql/penetration_rate.sql", "w") as f:
    f.write(sql)

penetration = con.execute(sql).df()

print("TAM VS ACTUAL ARR BY SUBREGION")
print(f"{'Subregion':<18} {'TAM':>14} {'Actual ARR':>12} {'Penetration%':>13} {'Whitespace ARR':>15} {'WS Accounts':>12}")
print("─" * 88)
for _, row in penetration.iterrows():
    print(f"{row['subregion']:<18} {row['tam_arr']:>14,.0f} {row['actual_arr']:>12,.0f} "
          f"{row['penetration_pct']:>12.2f}% {row['whitespace_arr']:>15,.0f} "
          f"{row['whitespace_accounts']:>12,.0f}")

TAM VS ACTUAL ARR BY SUBREGION
Subregion                     TAM   Actual ARR  Penetration%  Whitespace ARR  WS Accounts
────────────────────────────────────────────────────────────────────────────────────────
ANZ                   243,679,100    8,936,600         3.67%     132,709,800          279
North Asia            344,434,100    9,009,800         2.62%     225,003,600          618
SEA                   520,062,200   10,235,100         1.97%     361,389,100        1,016
Greater China         592,682,300   10,354,500         1.75%     435,299,100          893
India                 123,970,700    1,441,400         1.16%      99,800,000          656


## 3. Penetration Rate Chart

In [5]:
subregions = penetration["subregion"].tolist()
max_val = penetration["tam_arr"].max()

fig = go.Figure()

fig.add_trace(go.Bar(
    name="Whitespace ARR",
    x=subregions,
    y=penetration["tam_arr"],
    marker_color="#B0BEC5",
    text=[f"${v/1e6:.0f}M TAM" for v in penetration["tam_arr"]],
    textposition="outside"
))

fig.add_trace(go.Bar(
    name="Actual ARR",
    x=subregions,
    y=penetration["actual_arr"],
    marker_color="#2196F3",
    text=[f"{v:.1f}%" for v in penetration["penetration_pct"]],
    textposition="inside",
    textfont=dict(color="white", size=12)
))

fig.update_layout(
    title="TAM vs Actual ARR by Subregion — Penetration Rate",
    barmode="overlay",
    xaxis_title="Subregion",
    yaxis_title="ARR (USD)",
    yaxis=dict(range=[0, max_val * 1.35]),
    height=520,
    font=dict(size=13),
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
    plot_bgcolor="white",
    paper_bgcolor="white"
)

fig.show()

os.makedirs("../outputs", exist_ok=True)
fig.write_image("../outputs/03_penetration_rate.png", width=900, height=520, scale=2)
print("Saved: ../outputs/03_penetration_rate.png")

Saved: ../outputs/03_penetration_rate.png


## 4. K-Means Clustering — Whitespace Priority Tiers

In [6]:
# Prepare features for clustering
employee_band_map = {"1-50": 1, "51-200": 2, "201-500": 3, "501-2000": 4, "2001+": 5}
segment_map = {"SMB": 1, "Mid-Market": 2, "Enterprise": 3}

whitespace_ml = whitespace.copy()
whitespace_ml["employee_band_num"] = whitespace_ml["employee_band"].map(employee_band_map)
whitespace_ml["segment_num"] = whitespace_ml["segment"].map(segment_map)
whitespace_ml["estimated_arr"] = whitespace_ml["estimated_arr"].fillna(0)

features = ["estimated_arr", "employee_band_num", "segment_num"]
X = whitespace_ml[features].fillna(0)

# Elbow method
inertias = []
k_range = range(2, 9)

for k in k_range:
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("kmeans", KMeans(n_clusters=k, random_state=42, n_init=10))
    ])
    pipe.fit(X)
    inertias.append(pipe.named_steps["kmeans"].inertia_)

fig_elbow = go.Figure()
fig_elbow.add_trace(go.Scatter(
    x=list(k_range),
    y=inertias,
    mode="lines+markers",
    marker=dict(size=8, color="#2196F3"),
    line=dict(width=2)
))

fig_elbow.update_layout(
    title="Elbow Method — Optimal Number of Clusters",
    xaxis_title="Number of Clusters (k)",
    yaxis_title="Inertia",
    height=400,
    font=dict(size=13),
    plot_bgcolor="white",
    paper_bgcolor="white"
)

fig_elbow.show()
fig_elbow.write_image("../outputs/03_elbow_method.png", width=900, height=400, scale=2)
print("Saved: ../outputs/03_elbow_method.png")

Saved: ../outputs/03_elbow_method.png


In [7]:
final_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("kmeans", KMeans(n_clusters=3, random_state=42, n_init=10))
])

final_pipe.fit(X)
whitespace_ml["cluster"] = final_pipe.named_steps["kmeans"].labels_

# Label clusters by mean estimated_arr — highest ARR = Tier 1
cluster_means = whitespace_ml.groupby("cluster")["estimated_arr"].mean().sort_values(ascending=False)
tier_map = {cluster: f"Tier {i+1}" for i, cluster in enumerate(cluster_means.index)}
whitespace_ml["tier"] = whitespace_ml["cluster"].map(tier_map)

# Summary
tier_summary = (
    whitespace_ml.groupby("tier")
    .agg(
        n_accounts=("account_id", "count"),
        avg_arr=("estimated_arr", "mean"),
        total_arr=("estimated_arr", "sum")
    )
    .sort_values("avg_arr", ascending=False)
)

print("WHITESPACE TIERS — K-MEANS CLUSTERING")
print(f"{'Tier':<10} {'Accounts':>9} {'Avg ARR':>12} {'Total ARR':>14}")
print("─" * 50)
for tier, row in tier_summary.iterrows():
    print(f"{tier:<10} {row['n_accounts']:>9,} {row['avg_arr']:>12,.0f} {row['total_arr']:>14,.0f}")

WHITESPACE TIERS — K-MEANS CLUSTERING
Tier        Accounts      Avg ARR      Total ARR
──────────────────────────────────────────────────
Tier 1         882.0    1,156,456  1,019,994,500
Tier 2       1,235.0      159,959    197,548,800
Tier 3       1,345.0       27,255     36,658,300


In [8]:
tier_colors = {"Tier 1": "#1565C0", "Tier 2": "#42A5F5", "Tier 3": "#B0BEC5"}

fig_scatter = go.Figure()

for tier in ["Tier 1", "Tier 2", "Tier 3"]:
    mask = whitespace_ml["tier"] == tier
    fig_scatter.add_trace(go.Scatter(
        x=whitespace_ml.loc[mask, "employee_band_num"],
        y=whitespace_ml.loc[mask, "estimated_arr"],
        mode="markers",
        name=tier,
        marker=dict(
            color=tier_colors[tier],
            size=5,
            opacity=0.6
        ),
        text=whitespace_ml.loc[mask, "segment"],
        hovertemplate="<b>%{text}</b><br>Employee Band: %{x}<br>Est ARR: $%{y:,.0f}<extra></extra>"
    ))

fig_scatter.update_layout(
    title="K-Means Clustering — Whitespace Accounts by Priority Tier",
    xaxis=dict(
        title="Company Size (Employee Band)",
        tickvals=[1, 2, 3, 4, 5],
        ticktext=["1-50", "51-200", "201-500", "501-2000", "2001+"]
    ),
    yaxis=dict(title="Estimated ARR (USD)"),
    height=520,
    font=dict(size=13),
    plot_bgcolor="white",
    paper_bgcolor="white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02)
)

fig_scatter.show()
fig_scatter.write_image("../outputs/03_kmeans_scatter.png", width=900, height=520, scale=2)
print("Saved: ../outputs/03_kmeans_scatter.png")

Saved: ../outputs/03_kmeans_scatter.png


## 5. Whitespace Tiers by Subregion

In [10]:
tier_subregion = (
    whitespace_ml.groupby(["subregion", "tier"])
    .agg(
        n_accounts=("account_id", "count"),
        total_arr=("estimated_arr", "sum")
    )
    .reset_index()
    .sort_values(["subregion", "tier"])
)

# Pivot for easy reading
pivot = tier_subregion.pivot(index="subregion", columns="tier", values="n_accounts").fillna(0)
pivot_arr = tier_subregion.pivot(index="subregion", columns="tier", values="total_arr").fillna(0)

print("WHITESPACE ACCOUNTS BY TIER AND SUBREGION")
print(f"{'Subregion':<18} {'Tier 1':>8} {'Tier 2':>8} {'Tier 3':>8} {'Total':>8}")
print("─" * 48)
for subregion in pivot.index:
    t1 = int(pivot.loc[subregion, "Tier 1"]) if "Tier 1" in pivot.columns else 0
    t2 = int(pivot.loc[subregion, "Tier 2"]) if "Tier 2" in pivot.columns else 0
    t3 = int(pivot.loc[subregion, "Tier 3"]) if "Tier 3" in pivot.columns else 0
    total = t1 + t2 + t3
    print(f"{subregion:<18} {t1:>8,} {t2:>8,} {t3:>8,} {total:>8,}")

print()
print("WHITESPACE ARR ($) BY TIER AND SUBREGION")
print(f"{'Subregion':<18} {'Tier 1':>14} {'Tier 2':>14} {'Tier 3':>12}")
print("─" * 62)
for subregion in pivot_arr.index:
    t1 = pivot_arr.loc[subregion, "Tier 1"] if "Tier 1" in pivot_arr.columns else 0
    t2 = pivot_arr.loc[subregion, "Tier 2"] if "Tier 2" in pivot_arr.columns else 0
    t3 = pivot_arr.loc[subregion, "Tier 3"] if "Tier 3" in pivot_arr.columns else 0
    print(f"{subregion:<18} {t1:>14,.0f} {t2:>14,.0f} {t3:>12,.0f}")

WHITESPACE ACCOUNTS BY TIER AND SUBREGION
Subregion            Tier 1   Tier 2   Tier 3    Total
────────────────────────────────────────────────
ANZ                      93      145       41      279
Greater China           327      334      232      893
India                    56      178      422      656
North Asia              155      225      238      618
SEA                     251      353      412    1,016

WHITESPACE ARR ($) BY TIER AND SUBREGION
Subregion                  Tier 1         Tier 2       Tier 3
──────────────────────────────────────────────────────────────
ANZ                   107,390,200     24,219,100    1,100,500
Greater China         374,132,300     55,025,400    6,141,400
India                  60,847,400     27,373,700   11,578,900
North Asia            182,647,700     35,668,000    6,687,900
SEA                   294,976,900     55,262,600   11,149,600


## 6. Tier Distribution Chart

In [11]:
subregion_order = ["Greater China", "SEA", "North Asia", "India", "ANZ"]
tier_colors = {"Tier 1": "#1565C0", "Tier 2": "#42A5F5", "Tier 3": "#B0BEC5"}

fig_tiers = go.Figure()

for tier in ["Tier 1", "Tier 2", "Tier 3"]:
    values = [int(pivot.loc[s, tier]) if tier in pivot.columns else 0 for s in subregion_order]
    fig_tiers.add_trace(go.Bar(
        name=tier,
        x=subregion_order,
        y=values,
        marker_color=tier_colors[tier],
        text=values,
        textposition="inside",
        textfont=dict(color="white", size=12)
    ))

fig_tiers.update_layout(
    title="Whitespace Accounts by Priority Tier and Subregion",
    barmode="stack",
    xaxis_title="Subregion",
    yaxis_title="Number of Accounts",
    yaxis=dict(range=[0, 1200]),
    height=520,
    font=dict(size=13),
    plot_bgcolor="white",
    paper_bgcolor="white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02)
)

fig_tiers.show()
fig_tiers.write_image("../outputs/03_whitespace_tiers.png", width=900, height=520, scale=2)
print("Saved: ../outputs/03_whitespace_tiers.png")

Saved: ../outputs/03_whitespace_tiers.png


## 7. Priority Flag — Top Tier 1 Accounts

In [12]:
# Flag top 300 Tier 1 accounts by estimated_arr as Priority
tier1 = whitespace_ml[whitespace_ml["tier"] == "Tier 1"].copy()
top_n = 300

tier1_sorted = tier1.sort_values("estimated_arr", ascending=False)
priority_ids = set(tier1_sorted.head(top_n)["account_id"].tolist())

whitespace_ml["priority_flag"] = whitespace_ml["account_id"].apply(
    lambda x: "Priority" if x in priority_ids else "Standard"
)

# Summary
priority_summary = (
    whitespace_ml[whitespace_ml["priority_flag"] == "Priority"]
    .groupby("subregion")
    .agg(
        n_accounts=("account_id", "count"),
        total_arr=("estimated_arr", "sum"),
        avg_arr=("estimated_arr", "mean")
    )
    .sort_values("total_arr", ascending=False)
)

print(f"Total Priority accounts: {len(priority_ids):,} ({len(priority_ids)/len(whitespace_ml)*100:.1f}% of whitespace)")
print()
print("PRIORITY ACCOUNTS BY SUBREGION")
print(f"{'Subregion':<18} {'Accounts':>9} {'Total ARR':>14} {'Avg ARR':>12}")
print("─" * 57)
for subregion, row in priority_summary.iterrows():
    print(f"{subregion:<18} {row['n_accounts']:>9,} {row['total_arr']:>14,.0f} {row['avg_arr']:>12,.0f}")

Total Priority accounts: 300 (8.7% of whitespace)

PRIORITY ACCOUNTS BY SUBREGION
Subregion           Accounts      Total ARR      Avg ARR
─────────────────────────────────────────────────────────
Greater China          106.0    179,215,500    1,690,712
SEA                     89.0    153,728,900    1,727,291
North Asia              56.0     96,702,300    1,726,827
ANZ                     31.0     51,901,600    1,674,245
India                   18.0     31,008,200    1,722,678


In [13]:
os.makedirs("../data/processed", exist_ok=True)

whitespace_ml.to_csv("../data/processed/whitespace_scored.csv", index=False)
print(f"Saved: ../data/processed/whitespace_scored.csv")
print(f"Shape: {whitespace_ml.shape}")
print()
print("Columns:")
for col in whitespace_ml.columns:
    print(f"  {col}")

Saved: ../data/processed/whitespace_scored.csv
Shape: (3462, 17)

Columns:
  account_id
  company_name
  country
  subregion
  industry
  employee_band
  estimated_arr
  segment
  account_status
  coverage_status
  engagement_status
  last_activity_date
  employee_band_num
  segment_num
  cluster
  tier
  priority_flag


## Key Findings

1. **$1.25B whitespace ARR across APAC** — less than 4% of TAM captured in any subregion
2. **Greater China is the biggest prize:** 106 priority accounts, $179M ARR — large CN Enterprise completely untouched
3. **300 priority accounts identified (8.7% of whitespace):** avg ARR $1.7M — these warrant direct AE ownership
4. **India is a different problem:** 422 Tier 3 accounts dominate — SMB heavy, needs marketing-led nurture not direct rep coverage
5. **SEA has the most Tier 1 whitespace (251 accounts, $295M)** but reps are already overloaded — headcount addition has clearest ROI here
6. **Coverage model recommendation:** Tier 1 → AE direct, Tier 2 → BDR sequencing, Tier 3 → marketing campaigns